# 03-机械传动与位置反馈


## 1. 本篇范围与要解决的问题

上一笔记比较了舵机、步进电机和无刷电机。电机通常擅长高速旋转，而机械臂关节更需要**低转速、高扭矩和可测量的位置**。

本篇继续回答两个问题：

1. 怎样把电机的高速、低扭矩转换成关节需要的低速、高扭矩？
2. 机械臂怎样知道自己的关节实际转到了什么角度？

课程给出的答案分别是**机械传动**和**位置反馈**。传动系统负责改变速度与扭矩，位置传感器负责把关节状态反馈给控制器。两者共同决定机械臂的运动性能和闭环控制上限。

本篇依据 `具身智能实战01.pptx` 第 **77—101页**整理，覆盖减速比、功率与扭矩、齿隙、阻尼齿轮、电位器和磁编码器。第86页开始的内容进入位置反馈，第93页开始讨论数字磁编码器。


## 2. 从电机输出到机械臂关节

小型电机可以达到很高的转速，但直接用手就能限制它的轴。机械臂抬起连杆和负载时，通常更需要输出轴有足够的扭矩，而不是让关节高速空转。

```text
电机：高转速、低扭矩
          ↓ 传动系统
关节：低转速、高扭矩
```

传动系统可以通过齿轮、皮带、蜗轮蜗杆或其他机构改变运动状态。本课件先用齿轮减速说明基本关系，后续再把齿隙和阻尼齿轮作为工程问题展开。

### 2.1 为什么不能只看电机转速

只看电机的空载转速，无法判断关节能否带动负载。选型至少要同时关注：

| 参数 | 需要回答的问题 |
|---|---|
| 输出转速 | 关节运动是否足够快？减速后还剩多少速度？ |
| 输出扭矩 | 能否克服连杆、末端工具和负载产生的阻力矩？ |
| 传动效率 | 摩擦和损耗后实际还能输出多少？ |
| 回差与刚度 | 改变方向时，末端位置会不会出现空程和抖动？ |
| 反馈位置 | 控制器能否知道输出轴真实角度？ |

同一个电机配上不同减速比和不同传感器，可能形成完全不同的关节性能。传动和反馈不能从系统设计中单独拿掉。


## 3. 功率、扭矩与角速度

### 3.1 功率的基本关系

课件从第一性原理出发，使用旋转功率公式：

```text
P = τ × ω
```

- `P`：功率
- `τ`：扭矩
- `ω`：角速度

这个关系说明，在功率近似一定时，扭矩增大通常意味着角速度降低。减速器并不会凭空创造能量，而是把速度和扭矩重新分配。

### 3.2 理想齿轮系统中的能量守恒

理想情况下没有摩擦和其他损耗，输入功率约等于输出功率：

```text
P_in ≈ P_out
τ_in × ω_in ≈ τ_out × ω_out
```

整理后得到：

```text
τ_out / τ_in ≈ ω_in / ω_out
```

因此，输入和输出的扭矩比与角速度比成反比。实际传动还存在齿轮摩擦、轴承损耗、材料变形和发热，所以实际输出功率会小于输入功率。

### 3.3 一个数量级示例

假设电机输入扭矩为 `0.1 N·m`，输入角速度为 `1000 rpm`，减速后输出角速度为 `100 rpm`。理想减速比为：

```text
i = ω_in / ω_out = 1000 / 100 = 10
```

理想输出扭矩约为：

```text
τ_out ≈ τ_in × i = 0.1 × 10 = 1.0 N·m
```

如果传动效率只有 `80%`，可用输出扭矩还要进一步折减，不能直接按 `1.0 N·m` 设计负载。这个计算只用于理解趋势，真实选型还要考虑启动、加速、冲击和安全余量。


## 4. 减速比：牺牲速度换取力量

### 4.1 减速比的定义

课件定义减速比为：

```text
i = ω_in / ω_out
```

当 `i > 1` 时，输出速度低于输入速度，属于减速。忽略损耗时：

```text
τ_out ≈ τ_in × i
```

减速比越大，输出扭矩越大，但输出速度越低。工程设计不能只追求大减速比，否则会出现动作太慢、效率降低或传动结构体积增加等问题。

### 4.2 用汽车变速箱理解减速与增速

课件用汽车换挡作类比：

| 工况 | 齿轮关系 | 结果 |
|---|---|---|
| 低挡爬坡 | 减速比大于1 | 车轮转速低，但获得较大驱动力 |
| 高挡巡航 | 传动比更偏向增速 | 速度更高，但可用驱动力相对降低 |

机械臂关节通常需要在有限速度下获得较大扭矩，因此会使用减速传动。不同关节的负载和速度要求不同，减速比也不一定相同。

### 4.3 传动效率与额定值

理想公式适合解释方向，实际设计还要加入效率：

```text
P_out = η × P_in
τ_out ≈ η × τ_in × i
```

其中 `η` 是传动效率，取值小于1。效率会受到齿轮啮合、轴承、润滑、负载和转速影响。额定扭矩也不等于任何情况下都能持续输出的扭矩，需要区分持续、峰值和瞬时冲击工况。

**易错点：**减速器只能改变输出速度和扭矩，不能消除电机功率、散热和材料强度的限制。把减速比直接当作“无限放大力量”会导致错误选型。


## 5. 齿隙：改变方向时的空程问题

### 5.1 什么是齿隙

标准齿轮啮合时，齿与齿之间需要留出一定间隙，避免完全卡死并给加工误差、热膨胀和润滑留出空间。这个间隙就是齿隙（Backlash）。

当电机保持同一方向转动时，齿隙可能不明显；当机械臂启动、停止或改变方向时，齿轮需要先消除间隙，输出轴才会跟随反向运动。

```text
电机反向 → 先走完齿隙 → 输出轴才开始反向
```

这段没有有效输出的角度就是空程。它会造成：

- 末端定位误差；
- 启停和换向时的抖动；
- 运动不平顺；
- 齿轮碰撞产生的噪声。

### 5.2 齿隙为什么会影响控制

位置控制器根据目标角度和反馈角度计算误差。如果传动系统存在齿隙，电机轴已经转动，输出关节却可能没有立即移动。控制器看到的响应会出现延迟和非线性，容易出现反复修正、振荡或定位不一致。

齿隙还会随着磨损、装配误差、负载方向和温度变化而改变。因此，仅提高编码器分辨率，不能自动消除传动系统的回差。


## 6. 阻尼齿轮：用预紧力减少齿隙

### 6.1 结构与原理

课件给出的阻尼齿轮由两片薄齿轮和中间的弹簧组成。两片齿轮之间错开一个小角度，弹簧产生预紧力，使它们始终贴向配合齿轮的两个相对侧面。

```text
两片薄齿轮 + 弹簧预紧
        ↓
同时贴住配合齿的两侧
        ↓
减小换向时的空隙和冲击
```

它不是让齿轮间完全没有几何间隙，而是通过预紧力减少间隙带来的自由运动。预紧力过小，消隙效果有限；预紧力过大，则会增加摩擦、发热和电机负担。

### 6.2 工程价值与边界

课件总结阻尼齿轮可以：

| 改善方向 | 体现 |
|---|---|
| 定位精度 | 换向时减少空程，输出更接近目标 |
| 运动平顺性 | 减少启动、停止和反向时的抖动 |
| 噪声 | 减少齿面撞击和松旷产生的声音 |

阻尼齿轮不能替代正确的齿轮加工、装配、润滑和负载设计。它也会增加结构复杂度，弹簧和齿轮仍会随时间磨损。最终要在精度、摩擦、寿命和成本之间平衡。


## 7. 位置反馈：机器人的“本体感觉”

### 7.1 为什么必须测量关节位置

机械臂有了电机和传动，只能“产生力量”；控制器还需要知道这个力量是否让关节到达了目标。

位置反馈（Proprioception，本体感觉）描述机器人对自身关节状态的感知。它回答：

- 当前关节角度是多少？
- 输出轴是否真的跟上了电机命令？
- 负载或齿隙是否造成了位置误差？
- 发生外部扰动后，系统偏离了多少？

PID等闭环控制方法都需要当前值。没有位置反馈，控制器只能假设动作已经发生，无法根据真实误差修正。

### 7.2 反馈链路

```text
关节转动
  → 传感器把机械状态转换成电信号
  → 控制器采集并转换成数字值
  → 软件根据标定关系得到角度
  → 与目标角度比较，产生下一步控制量
```

传感器输出不一定就是角度。电位器输出电压，编码器可能输出总线角度或脉冲，控制器需要根据对应的信号类型、标定关系和零位才能得到可用的关节角度。


## 8. 电位器：低成本的模拟位置传感器

### 8.1 可变电阻原理

电位器是一个可变电阻器。滑动触点沿电阻材料移动时，接入电路的电阻比例变化，输出端电压随之变化。

```text
[输入端] ─── 电阻材料 ─── [输出端]
                 ↑
             滑动触点
```

在理想线性电位器和合适的接线方式下，滑动触点的位置与分压比例相关，因此可以用电压表示机械角度。

### 8.2 分压电路

课件给出的分压关系为：

```text
V_out = V_in × (R_part / R_total)
```

- `V_in`：电位器两端的输入电压；
- `R_total`：总电阻；
- `R_part`：滑动触点到某一端之间的电阻；
- `V_out`：滑动端输出电压。

如果电阻材料均匀、机械角度范围和电压范围已经标定，那么可以近似认为角度随 `V_out` 线性变化。实际电位器可能存在非线性，不能不经标定就直接把电压当作精确角度。

### 8.3 从机械角度到数字值

课件把转换过程表示为：

```text
机械角度
  → 电阻值变化
  → 模拟电压（例如0—5V）
  → ADC采样
  → 数字值（例如0—1023）
  → 根据标定关系反推出角度
```

ADC（Analog-to-Digital Converter，模数转换器）把模拟电压转换成数字量。以10位ADC为例，理想输出码范围通常是 `0—1023`，但实际可用范围还受参考电压、输入噪声、采样电路和校准影响。

### 8.4 电位器的优点与局限

| 优点 | 局限 |
|---|---|
| 成本低 | 模拟信号容易受到电源和电磁干扰 |
| 电路简单 | 滑动触点接触式工作，会摩擦和磨损 |
| 技术成熟 | 廉价器件线性度可能较差 |
| 容易接入ADC | 分辨率受ADC位数和噪声限制 |

普通SG90舵机中常见电位器。课件用这个例子说明：低成本模拟传感器的性能会影响舵机的精度和寿命。电位器适合入门、教学和成本敏感的方案，但在高精度、高可靠性关节中会遇到明显上限。


## 9. 磁编码器：非接触式数字位置反馈

### 9.1 从模拟传感到数字传感

磁编码器通过检测旋转磁场来测量角度。它不需要滑动触点，能够避免电位器的接触磨损和部分模拟干扰问题。

一个典型磁编码器包含：

| 组件 | 安装与作用 |
|---|---|
| 径向充磁磁铁 | 固定在电机轴或传动轴末端，随轴旋转 |
| 传感器芯片 | 固定在电路板上，位于磁铁附近，检测磁场方向或变化 |

磁铁和芯片的相对位置很关键。对中误差、轴向间距和倾斜都会影响磁场测量，非接触并不等于不需要精确装配。

### 9.2 磁场角度到数字信号

当磁铁旋转时，传感器检测磁场方向变化。课件简要提到霍尔效应：带电粒子在磁场中运动会受到洛伦兹力影响，导体两侧形成微小电压差。芯片内部的信号处理电路根据检测结果计算磁场方向，并输出数字信号。

工程上不必把“霍尔效应”简单理解为直接读出机械角度。芯片还包含磁场检测、信号处理、角度计算和接口输出等环节，最终角度精度取决于芯片、磁铁、安装和校准。

### 9.3 绝对式与增量式

课件把磁编码器的输出分成两类。

| 类型 | 输出方式 | 优点 | 需要注意 |
|---|---|---|---|
| 绝对式（Absolute） | 通过SPI、I²C等总线直接报告当前角度，例如 `135.7°` | 上电即可知道位置，通常不必先回零 | 需要正确读取协议、处理零位和角度范围 |
| 增量式（Incremental） | 输出A/B相正交脉冲，控制器通过计数与相位差计算位移和方向 | 电路相对简单、速度快 | 掉电后累计位置通常丢失，常需要回零 |

A/B相正交信号的相位关系可以帮助判断旋转方向，但具体计数倍频、脉冲分辨率和接口电平要以编码器资料为准。

### 9.4 磁编码器的优点与局限

| 优点 | 局限 |
|---|---|
| 非接触，磨损小，寿命长 | 成本通常高于电位器 |
| 数字信号，抗干扰能力较强 | 对磁铁对中、间距和安装精度有要求 |
| 分辨率和精度潜力较高 | 仍需处理零位、校准、接口和采样延迟 |
| 可以适应较高转速 | 磁场环境和机械结构可能影响测量 |

课件将磁编码器与高端舵机、工业机器人关节和无人机云台联系起来，强调它是高精度、高动态响应控制的重要基础。


## 10. 电位器与磁编码器的选型比较

| 特性 | 电位器 | 磁编码器 |
|---|---|---|
| 信号 | 模拟电压 | 数字角度、总线数据或脉冲 |
| 工作方式 | 接触式 | 非接触式 |
| 成本 | 低 | 较高 |
| 电路 | 简单，接ADC即可读取 | 需要接口、计数或协议解析 |
| 磨损 | 有 | 传感部分无接触磨损 |
| 抗干扰 | 取决于模拟布线和滤波 | 数字传输通常更稳健，但仍需正确接地和布线 |
| 精度潜力 | 受线性度、ADC和磨损影响 | 受芯片、磁铁、安装和校准影响，通常更高 |
| 掉电后的位置信息 | 由当前电压表示，重新上电可读，但要正确建立零位 | 绝对式可直接读当前位置；增量式通常需回零 |
| 适合场景 | 教学、原型、成本敏感设备 | 高性能关节、高可靠性和高动态响应系统 |

选型时不能只看“模拟”或“数字”标签。电位器方案也可以通过良好电路和标定完成稳定工作，磁编码器也需要正确安装和校准。真正的判断依据是任务要求、允许误差、寿命、环境和维护成本。


## 11. 传动与反馈如何共同影响闭环控制

机械臂关节的真实控制链可以概括为：

```text
目标角度
  → 控制器
  → 电机
  → 减速传动
  → 关节输出
  → 位置传感器
  → 当前角度反馈
  → 误差修正
```

传动和反馈分别影响不同问题：

| 问题 | 主要相关因素 |
|---|---|
| 能否带动负载 | 电机扭矩、减速比、传动效率、机械强度 |
| 动作是否足够快 | 电机转速、减速比、负载惯量、加速度限制 |
| 换向是否平顺 | 齿隙、预紧力、齿轮加工和控制参数 |
| 控制器是否知道真实位置 | 电位器、磁编码器、编码器安装位置和标定 |
| 位置是否稳定可靠 | 反馈分辨率、噪声、传动回差、采样与控制延迟 |

**高分辨率反馈不能修复所有机械问题。**如果输出端存在明显齿隙，而编码器只安装在电机输入轴上，控制器可能看到输入轴已经到位，却没有准确测到输出端的回差。反馈安装位置需要结合控制目标决定。


## 12. 工程选型的权衡

课件在最后强调，传动和传感器同样体现“工程的权衡艺术”。

| 使用阶段 | 主要目标 | 课程给出的倾向 |
|---|---|---|
| 教学与原型验证 | 快速实现、成本可控、结构易理解 | 低成本舵机和电位器方案更容易入门 |
| 产品级应用 | 高精度、长寿命、抗干扰和可靠运行 | 编码器和更完整的伺服反馈方案更合适 |

这不是“低成本方案永远不能用于产品”，也不是“编码器装上就一定解决问题”。工程决策还要看任务是否允许回零、负载变化是否稳定、是否要求高速、高动态响应，以及维护和替换条件。

可以按下面的顺序做初步判断：

1. 先写出关节的最大负载、目标速度和允许定位误差。
2. 根据电机和减速比估算输出扭矩与速度，并加入效率和安全余量。
3. 评估齿隙、结构刚度、噪声和换向抖动是否满足任务。
4. 决定反馈需要测量电机轴还是输出轴，以及是否必须上电即知位置。
5. 在电位器、增量编码器和绝对式编码器之间比较成本、接口、回零和寿命。
6. 最后再核对控制器资源、供电、布线、散热和维护条件。


## 13. 课件要点回顾

1. 电机通常提供高转速、低扭矩输出，机械臂关节通过减速传动获得低速、高扭矩。
2. 旋转功率满足 `P = τ × ω`。理想减速比 `i = ω_in / ω_out`，输出扭矩约为输入扭矩乘以减速比，实际还要扣除传动损耗。
3. 齿隙会在启动、停止和换向时造成空程、抖动、噪声和定位误差。
4. 阻尼齿轮通过两片齿轮和弹簧预紧力减少齿隙影响，但会增加摩擦、结构复杂度和维护要求。
5. 位置反馈是闭环控制的基础。电位器把角度变化转换成模拟电压，再由ADC转换成数字量。
6. 电位器成本低、电路简单，但容易受到磨损、干扰、线性度和ADC分辨率限制。
7. 磁编码器采用磁铁与传感器芯片进行非接触测量，通常具有更高的精度和寿命，但对安装和成本有更高要求。
8. 绝对式编码器上电即可读取位置；增量式编码器通常需要计数，并在掉电后回零。

下一篇进入数字孪生、仿真与Sim-to-Real，讨论为什么机器人应先在虚拟环境中开发、测试和生成数据。


## 14. 来源与覆盖范围

资料来自[用户提供的课程网盘](https://pan.baidu.com/s/1JTzTYgpIAiSUroQ2-ihJHA?pwd=1234)，提取码为 `1234`。资料路径为“6、嵌入式具身智能机械臂／视频资料／课程ppt”。

| 笔记章节 | 对应课件页码 |
|---|---|
| 从电机到传动 | 77—78 |
| 功率、减速比与扭矩 | 79—82 |
| 齿隙与阻尼齿轮 | 83—85 |
| 位置反馈与电位器 | 86—92 |
| 磁编码器与信号类型 | 93—99 |
| 电位器、磁编码器和工程权衡 | 100—101 |

本篇未转录配套视频；内容以课件PPT为主，并在公式、效率、ADC和反馈安装位置处补充了理解边界。
